In [78]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [79]:
artists = pd.read_json('artists.jsonl', lines=True)
genres_to_filter = ['pop', 'trap', 'rap', 'rock', 'metal', 'hip hop']

In [80]:
def filter_genres(genre_list, regex_pattern):
    return [genre for genre in genre_list if pd.Series([genre]).str.contains(regex_pattern, regex=True).any()]


In [81]:
filtered_artists = pd.DataFrame()
filtered_genres_dict = {}
for genre in genres_to_filter:
    regex_pattern = fr'\b({genre})\b'

    artists['filtered_genres'] = artists['genres'].apply(lambda x: filter_genres(x, regex_pattern))
    one_genre = artists[artists['filtered_genres'].apply(lambda x: x != [])]
    one_genre.loc[:, 'genres'] = genre
    filtered_artists = pd.concat([filtered_artists, one_genre], ignore_index=True)

# Drop the temporary 'filtered_genres' column
filtered_artists = filtered_artists.drop(columns=['filtered_genres'])

# Display the concatenated DataFrame
print(filtered_artists)

/tmp/ipykernel_714/77786420.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return [genre for genre in genre_list if pd.Series([genre]).str.contains(regex_pattern, regex=True).any()]
/tmp/ipykernel_714/77786420.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return [genre for genre in genre_list if pd.Series([genre]).str.contains(regex_pattern, regex=True).any()]


                          id                name   genres
0     7jVv8c5Fj3E9VhNjxT4snq           Lil Nas X      pop
1     0xRXCcSX89eobfrshSVdyu              MEDUZA      pop
2     4f7KfxeHq9BiylGmyXepGt      Tanishk Bagchi      pop
3     56mfhUDKa1vec6rSLZV5Eg           Jawsh 685      pop
4     6m6e7D2TnV0aYMllFFwMxu  Guilherme & Benuto      pop
...                      ...                 ...      ...
2277  50co4Is1HCEo8bhOyUWKpn          Young Thug  hip hop
2278  6U3ybJ9UHNKEdsH7ktGBZ7                 JID  hip hop
2279  1qQLhymHXFPtP5U8KNKsm6          Apache 207  hip hop
2280  6qgnBH6iDM91ipVXv28OMu          KAYTRANADA  hip hop
2281  46MWeeHNVMYRIIofQBEX98                BEAM  hip hop

[2282 rows x 3 columns]


In [82]:
len([ x for x in filtered_artists['genres'] if x == 'trap'])

268

In [83]:
filtered_artists.describe()

,id,name,genres
count,2282,2282,2282
unique,1391,1391,6
top,5gCRApTajqwbnHHPbr2Fpi,Juicy J,pop
freq,4,4,919


In [84]:
tracks = pd.read_json('tracks.jsonl', lines=True)
tracks.sample()

,id,name,popularity,duration_ms,explicit,id_artist,release_date,danceability,energy,key,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
18647,7fURZRPkB2S70sYR1naKTK,Black Hole Sun,51,318000,0,5xUf6j4upBrXZPg6AI4MRK,1994-01-01,0.361,0.728,6,-8.519,0.0368,0.000291,0.000112,0.0701,0.155,105.768


In [85]:
divided_songs = {}
for genre in genres_to_filter:
    genre_frame = filtered_artists[filtered_artists['genres'] == genre]
    genre_artist_ids = genre_frame['id'].tolist()
    divided_songs[genre] = tracks[tracks['id_artist'].apply(lambda x: x in genre_artist_ids)]

In [86]:
divided_songs['pop']

,id,name,popularity,duration_ms,explicit,id_artist,release_date,danceability,energy,key,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
29,1XPj5quoeV5Gd0paSUDpvm,You Send Me - Mono,61,165733,0,6hnWRPzGGKiapVX1UCdEAC,1958-02-01,0.572,0.365,4,-7.583,0.0276,0.8800,0.00000,0.1250,0.4100,96.022
45,2G0GextMwZJLkNxcSZ7ZJ3,(What A) Wonderful World - Mono,67,128787,0,6hnWRPzGGKiapVX1UCdEAC,1960-02,0.686,0.672,11,-5.523,0.0323,0.7000,0.00000,0.1350,0.8570,128.550
46,4yjz1aazw6R8ZURpGbCAkp,Bring It On Home To Me,65,165280,0,6hnWRPzGGKiapVX1UCdEAC,1960-01-01,0.482,0.569,0,-5.897,0.0328,0.6170,0.00000,0.1550,0.6090,70.843
48,0BFEyqJ9DJXS7gKg0Kj46R,You Send Me,64,165560,0,6hnWRPzGGKiapVX1UCdEAC,1960-01-01,0.553,0.291,4,-10.426,0.0301,0.8780,0.00000,0.1290,0.4070,96.217
50,5Fih0qlqrBjPIE0dFamuBr,(What A) Wonderful World,62,129293,0,6hnWRPzGGKiapVX1UCdEAC,1960-01-01,0.763,0.571,11,-10.552,0.0337,0.5290,0.00000,0.1540,0.9360,128.211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22407,0LcNMuOiULmxJK3bdHTfDF,Missing You,65,214987,0,7gAppWoH7pcYmphCVTXkzs,2019-04-19,0.537,0.647,2,-7.370,0.0519,0.1680,0.00000,0.1080,0.5310,79.770
22408,1uviKYHZuM4uINK33F7sCt,Fix It to Break It,70,198799,0,7okSU80WTrn4LXlyXYbX3P,2020-03-27,0.493,0.461,2,-8.524,0.0456,0.8450,0.00000,0.1150,0.3500,51.414
22409,1fXmDeiCb3ABt5CzkMxp4u,Lotus Inn,63,195868,0,2jnIB6XdLvnJUeNTy5A0J2,2020-12-04,0.578,0.740,2,-5.080,0.0721,0.0119,0.00000,0.1800,0.4950,155.984
22410,27kcZEJvhkb1rzZS9gCpdA,remember the mornings,67,202355,0,7okSU80WTrn4LXlyXYbX3P,2020-11-27,0.590,0.443,0,-5.662,0.0722,0.6180,0.00000,0.1100,0.3420,149.447


## Wyliczanie współczynnika dla każdej piosenki względem sesji

Ile razy dana piosenka znalazła się w jednej sesji z innymi piosenkami zakwalifikowanymi do tej samej grupy gatunkowej.
Nie bierzemy pod uwagę piosenek przewiniętych "skip".

In [87]:
# sessions = pd.read_json('sessoins_part1.jsonl', lines=True)
# tracks.sample()

In [88]:
import pandas as pd
import json

filename = 'sessions_part1.jsonl'

# Read and clean the JSONL file
cleaned_lines = []
with open(filename, 'r', encoding='utf-8') as file:
    for line_number, line in enumerate(file, start=1):
        line = line.strip()
        if line:
            try:
                json_obj = json.loads(line)
                cleaned_lines.append(json_obj)
            except json.JSONDecodeError as e:
                print(f"Error parsing JSON at line {line_number}: {e}")
                break

# Create a DataFrame from the cleaned lines
sessions = pd.DataFrame(cleaned_lines)


In [89]:
sessions = sessions[sessions['event_type'] == 'play']


In [90]:
for genre in genres_to_filter:
    divided_songs[genre].loc[:, 'coherence'] = 0


/tmp/ipykernel_714/422349128.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  divided_songs[genre].loc[:, 'coherence'] = 0
/tmp/ipykernel_714/422349128.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  divided_songs[genre].loc[:, 'coherence'] = 0
/tmp/ipykernel_714/422349128.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

In [91]:
def collect_track_ids(track_id, tracks_in_session, tracks_in_session_collected):
    if track_id in tracks_in_session:
        tracks_in_session_collected.append(track_id)

In [100]:
sessions_ids = set(sessions['session_id'][:100000].tolist())
for session_id in sessions_ids:
    tracks_in_session = sessions[sessions['session_id'] == session_id]['track_id'].tolist()
    for genre, track_list in divided_songs.items():
        tracks_in_session_collected = []
        track_list['id'].apply(lambda x: collect_track_ids(x, tracks_in_session, tracks_in_session_collected))
        if len(tracks_in_session) > 1:
            for track_id in tracks_in_session_collected:
                track_list.loc[track_list['id'] == track_id, 'coherence'] += 1

In [101]:
divided_songs['pop']['coherence']#.apply(lambda x: print(x) if x > 3 else print('', end=''))

29        0
45        1
46       11
48        4
50        1
         ..
22407     4
22408     9
22409     0
22410     7
22411     4
Name: coherence, Length: 11637, dtype: int64

In [104]:
numeric_features = divided_songs['metal'].select_dtypes(include=['int64', 'float64']).columns
covariance_matrix = divided_songs['metal'][numeric_features].corr()
covariance_matrix.style.background_gradient(cmap='coolwarm', axis=None, vmin=-1, vmax=1).format(precision=3)

,popularity,duration_ms,explicit,danceability,energy,key,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,coherence
popularity,1.000,-0.059,-0.023,0.084,0.007,-0.005,0.064,-0.058,0.009,-0.081,-0.031,0.095,-0.018,0.119
duration_ms,-0.059,1.000,-0.115,-0.230,-0.212,0.023,-0.266,-0.086,-0.014,0.240,-0.084,-0.261,-0.054,-0.010
explicit,-0.023,-0.115,1.000,0.049,0.162,0.038,0.088,0.275,-0.108,-0.018,0.039,0.058,0.011,-0.062
danceability,0.084,-0.230,0.049,1.000,-0.063,-0.007,0.038,-0.198,0.048,-0.092,-0.010,0.401,-0.277,-0.024
energy,0.007,-0.212,0.162,-0.063,1.000,0.007,0.710,0.393,-0.573,-0.167,0.073,0.234,0.131,-0.004
key,-0.005,0.023,0.038,-0.007,0.007,1.000,-0.043,-0.007,0.002,0.058,-0.050,-0.005,-0.026,0.009
loudness,0.064,-0.266,0.088,0.038,0.710,-0.043,1.000,0.229,-0.445,-0.340,0.061,0.118,0.108,0.018
speechiness,-0.058,-0.086,0.275,-0.198,0.393,-0.007,0.229,1.000,-0.149,-0.071,0.058,-0.047,0.156,0.012
acousticness,0.009,-0.014,-0.108,0.048,-0.573,0.002,-0.445,-0.149,1.000,0.017,0.001,-0.082,-0.093,0.029
instrumentalness,-0.081,0.240,-0.018,-0.092,-0.167,0.058,-0.340,-0.071,0.017,1.000,-0.048,-0.121,-0.040,-0.022


In [127]:
divided_songs['metal'].loc[:, ('popularity', 'duration_ms', 'danceability',
                                                     'energy', 'loudness', 'speechiness', 'acousticness', 'liveness',
                                                     'valence', 'tempo')]

,popularity,duration_ms,danceability,energy,loudness,speechiness,acousticness,liveness,valence,tempo
360,60,266840,0.570,0.635,-10.231,0.0482,0.004590,0.0901,0.196,108.431
490,78,168440,0.429,0.631,-11.514,0.0769,0.000078,0.1750,0.376,162.734
500,72,354774,0.332,0.769,-11.503,0.0765,0.003670,0.0372,0.525,157.378
518,66,474400,0.464,0.612,-12.019,0.0591,0.116000,0.1710,0.496,91.499
532,63,364493,0.476,0.556,-10.586,0.0646,0.083800,0.1250,0.430,103.122
...,...,...,...,...,...,...,...,...,...,...
22329,53,196813,0.654,0.660,-5.594,0.0283,0.025200,0.2780,0.412,108.039
22331,66,244573,0.357,0.549,-7.843,0.0698,0.000798,0.2200,0.531,130.749
22341,52,268333,0.509,0.883,-4.640,0.0403,0.021800,0.2760,0.590,89.965
22352,54,350644,0.163,0.977,-3.585,0.0591,0.000036,0.0555,0.369,178.440


In [131]:
from sklearn.feature_selection import mutual_info_regression

# mutual_info_regression(divided_songs['rock'].loc[:, ('popularity', 'duration_ms', 'danceability',
#                                                      'energy', 'loudness', 'speechiness', 'acousticness', 'liveness',
#                                                      'valence', 'tempo')], divided_songs['rock']['coherence'])
mutual_info_regression(divided_songs['trap'].loc[:, ('popularity', 'duration_ms', 'danceability',
                                                     'energy', 'loudness', 'speechiness', 'acousticness', 'liveness',
                                                     'valence', 'tempo')], divided_songs['trap']['coherence'])

array([0.00361141, 0.01818133, 0.        , 0.        , 0.00252836,
       0.        , 0.00918199, 0.02715217, 0.        , 0.        ])